# Phase 5 — RAG (Retrieval-Augmented Generation) with Citations

**Goal**: An LLM only knows what it was trained on — it does **not** know *your* business's definitions. Ask it *"what does 'churned' mean in our analysis?"* and it gives a generic textbook answer, not your company's actual rule (*Recency > 180 days*).

**RAG** fixes that. Before the agent answers, you **retrieve** the relevant reference material from your own knowledge corpus and inject it into the prompt. The agent answers **grounded** in that material — and **cites** where each fact came from.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Corpus** | Your knowledge base — here, the business definitions in `data/definitions.md`. |
| **Retrieval** | Finding the few documents most relevant to a question. We use **TF-IDF** — a fast, classic keyword-overlap method. |
| **Augmented prompt** | The retrieved documents, pasted into the prompt *before* the question. |
| **Grounding** | The agent answers using *only* the provided material — not its own guesses. |
| **Citation** | The agent tags each fact with the source it came from, e.g. `[Churn]`. |

## Acceptance criteria
1. The retriever surfaces the relevant definition for the question
2. The agent's answer contains the key fact from *your* corpus (not a generic guess)
3. The agent **cites** its source
4. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
from orchestrator.observability import Trace, Timer
from orchestrator.agents import run_agent
# Phase 5 adds the RAG module:
from orchestrator.rag import load_definitions, Retriever, format_context
from claude_agent_sdk import ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")
print(f"Dev model: {DEV_MODEL}")

## 2. The knowledge corpus

Your corpus is `data/definitions.md` — a glossary of business terms (VIP, Churn, RFM, Revenue, ...). `load_definitions()` splits it into one **Document** per term. Each Document has a `doc_id` (the term), which is what the agent will cite.

In [ ]:
documents = load_definitions("data/definitions.md")

print(f"Loaded {len(documents)} documents from the corpus:")
for doc in documents:
    print(f"  [{doc.doc_id}]")

## 3. The retriever — TF-IDF

`Retriever` builds a **TF-IDF** model over the corpus. Given a query, `retrieve()` returns the *k* most relevant documents, each with a similarity score (higher = more relevant).

Run the cell to see which documents it pulls for a churn-related query.

In [ ]:
retriever = Retriever(documents)

demo_query = "what does it mean for a customer to be churned"
print(f"Query: {demo_query!r}")
print()
print("Top 3 retrieved documents:")
for doc, score in retriever.retrieve(demo_query, k=3):
    print(f"  [{doc.doc_id}]  similarity = {score:.3f}")

## 4. Why RAG — the agent doesn't know *your* definitions

Watch the problem first. We ask a plain agent — **no retrieval** — what "churned" means *in our analysis*. It answers from general knowledge, and it does **not** know your specific rule (Recency > 180 days).

In [ ]:
plain_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt="You are a helpful analytics assistant.",
    max_turns=2,
)

demo_run = await run_agent(
    "PlainAgent",
    "In our customer analysis, what does it mean for a customer to be 'churned'?",
    plain_options,
)
print("----- Plain agent answer (no retrieval) -----")
print(demo_run.answer)
print()
print("^ A generic answer. It does NOT know OUR specific rule: Recency > 180 days.")

## 5. The RAG agent

The RAG agent's job: answer using **only** the reference material it's handed, and **cite** every fact. This discipline — grounding + citation — is the whole point of RAG. It's controlled entirely by the system prompt.

**TODO 1**: write the RAGAgent's system prompt.

In [ ]:
rag_system_prompt = (
    # ----- TODO 1: write the RAGAgent's system prompt --------------------
    "You are a grounded analytics assistant. Answer every question using "
    "ONLY the REFERENCE MATERIAL provided in the prompt. Do NOT use your "
    "own outside or general knowledge — if a fact is not in the reference "
    "material, you do not know it. "
    "After every fact you state, cite its source by appending the document's "
    "bracketed id (e.g. [Churn], [VIP]) on the same line. Use multiple "
    "citations when a sentence draws from multiple documents. "
    "If the reference material does not contain enough information to answer "
    "the question, say so plainly — something like 'The reference material "
    "does not define this' — rather than guessing or inventing a definition."
    # ---------------------------------------------------------------------
)

rag_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=rag_system_prompt,
    max_turns=2,
)
print("[OK] RAGAgent configured")

## 6. Answer with RAG

`answer_with_rag()` is the **retrieve -> augment -> generate** pipeline:

1. **Retrieve** the top-k documents for the question
2. **Augment** — format them into a context block, prepend to the question
3. **Generate** — run the RAG agent on the augmented prompt

**TODO 2**: wire up the retrieve + augment steps.

In [ ]:
async def answer_with_rag(question, retriever, rag_options, k=3):
    """The retrieve -> augment -> generate pipeline."""

    # ----- TODO 2: retrieve docs and build the augmented prompt ----------
    retrieved = retriever.retrieve(question, k=k)
    context = format_context(retrieved)
    prompt = context + "\n\nQuestion: " + question
    # ---------------------------------------------------------------------

    rag_run = await run_agent("RAGAgent", prompt, rag_options)
    return {"answer": rag_run.answer, "retrieved": retrieved, "run": rag_run}

print("[OK] answer_with_rag defined")

## 7. Run it

**TODO 3**: write the question. Ask what it means for a customer to be **'churned'** in our analysis — the corpus has a specific rule for this.

In [ ]:
# ----- TODO 3: write your question -----
QUESTION = "In our customer analysis, what does it mean for a customer to be 'churned'?"
# ---------------------------------------

trace = Trace(model=DEV_MODEL, question=QUESTION)
with Timer() as t:
    result = await answer_with_rag(QUESTION, retriever, rag_options, k=3)

r = result["run"]
trace.input_tokens        = r.input_tokens
trace.output_tokens       = r.output_tokens
trace.cached_input_tokens = r.cached_input_tokens
trace.n_tool_calls        = r.n_tool_calls
trace.n_delegations       = 1
trace.latency_ms          = t.elapsed_ms
trace.answer              = result["answer"]
trace.compute_cost()

print("----- Retrieved for this question -----")
for doc, score in result["retrieved"]:
    print(f"  [{doc.doc_id}]  similarity = {score:.3f}")
print()
print("----- RAG answer -----")
print(result["answer"])

## 8. Verify (acceptance criteria)

In [ ]:
# ----- TODO 4: write the checks -----
has_fact = "180" in result["answer"]
has_citation = "[" in result["answer"]
# ------------------------------------

trace.passed = bool(has_fact and has_citation)
print(f"Answer contains the key fact ('180'): {has_fact}")
print(f"Answer cites a source ('['):          {has_citation}")
print(f"Passed: {trace.passed}")
assert trace.passed, "RAG answer must contain the 180-day fact AND cite a source."

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - grounded answer with citation, trace saved")

## 9. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer in your own words.

1. **Why RAG**: the plain agent in section 4 gave a generic answer about "churned". Why didn't it know your specific 180-day rule? What does retrieval add that training alone cannot?

   The model was trained on the public internet — your `definitions.md` was never in that training set, so it can only fall back to a textbook definition ("a customer who stopped buying"). Retrieval brings *your* private knowledge into the prompt at inference time, so the answer reflects your company's rule, not a generic average of every customer-analytics blog post on the web.

2. **Grounding**: the system prompt tells the agent to answer using ONLY the reference material. Why is that instruction important — what bad behaviour does it prevent?

   It prevents **mixing** — the model interleaving its memorized generic knowledge with the retrieved snippets, producing an answer that sounds authoritative but quietly contradicts your real definition. Without grounding the model might say "churned customers are those who haven't bought in 90 days [Churn]" — confidently citing your doc while reporting a number from training data. Grounding makes the corpus the *only* source of truth.

3. **Citations**: the agent tags facts with `[Churn]` etc. Why do citations matter for a business analytics assistant? (Think about a user who doubts an answer.)

   Citations make the answer **auditable**. A skeptical user can jump straight to the cited definition and verify it themselves; without that, every claim is "trust me." Citations also turn the agent from a black box into a research assistant — same workflow as Wikipedia footnotes or a court brief.

4. **TF-IDF retrieval**: retrieval ranks documents by keyword overlap with the query. Give one kind of question where keyword matching might retrieve the *wrong* document — and what you'd upgrade to (hint: the README mentions chromadb / embeddings).

   Synonyms and paraphrases break TF-IDF: "lapsed buyers" should match the **Churn** doc but shares zero keywords with it, so TF-IDF surfaces the wrong thing. Upgrade to **embedding-based retrieval** (chromadb + a sentence-transformer) — embeddings score *semantic* similarity, so "lapsed", "inactive", "stopped purchasing" all map close to "churned" in vector space.

5. **The corpus is the limit**: if a user asks something the corpus does not cover, what *should* a well-prompted RAG agent do? Why is that better than letting it answer from general knowledge?

   It should say plainly *"the reference material doesn't define this"* and stop. That's better than guessing because (a) silence is safer than confident wrongness — a wrong answer poisons downstream decisions, and (b) it surfaces a **corpus gap** you can fix: every "I don't know" is a signal to add a definition. Quiet hallucination hides the gap; honest refusal exposes it.

## Phase 5 done when:
- [x] TODO 1 (RAGAgent system prompt) filled in
- [x] TODO 2 (retrieve + augment pipeline) filled in
- [x] TODO 3 (your question) filled in
- [x] TODO 4 (the two checks) filled in
- [x] TODO 5 (quiz) answered
- [x] Cell 9 shows the plain agent giving a generic (non-corpus) answer
- [x] Cell 15 shows a grounded answer citing a source
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has a new line

Then ping me — we'll review your work, then move to Phase 6 (MCP server).